# 📈 Phase 4: Exploratory Data Analysis (EDA) Notebook
### Real-Time Stock Market Analytics & ML Forecasting System

This notebook serves as the exploratory workspace to inspect the stock price datasets, analyze correlations across multiple sectors, study daily returns distributions, and visualize historical volatility trends.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for figures
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

## 1. Load Combined Processed Datasets

In [ ]:
processed_dir = "../data/processed"
tickers = ["AAPL", "TSLA", "MSFT", "RELIANCE_NS", "TCS_NS"]

all_dfs = []
for t in tickers:
    file_path = f"{processed_dir}/{t}_historical_processed.csv"
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df['Date'] = pd.to_datetime(df['Date'])
        all_dfs.append(df)
        print(f"Loaded {len(df)} rows for ticker: {df['Ticker'].iloc[0]}")
    else:
        # Check if running from root
        file_path_alt = f"data/processed/{t}_historical_processed.csv"
        if os.path.exists(file_path_alt):
            df = pd.read_csv(file_path_alt)
            df['Date'] = pd.to_datetime(df['Date'])
            all_dfs.append(df)
            print(f"Loaded {len(df)} rows for ticker: {df['Ticker'].iloc[0]}")
        else:
            print(f"Warning: Processed file not found for {t}")

combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df.head()

## 2. Dynamic Price Movement Trends
Pivot close prices and calculate relative normalized growth (Base = 100) to contrast historical performances.

In [ ]:
pivot_close = combined_df.pivot(index='Date', columns='Ticker', values='Close').ffill().bfill()
normalized_growth = (pivot_close / pivot_close.iloc[0]) * 100

normalized_growth.plot()
plt.title("Normalized Historical Stock Price Trends (Day 1 Base = 100)")
plt.ylabel("Relative Growth (%)")
plt.xlabel("Timeline")
plt.legend(title="Equities")
plt.grid(True)
plt.show()

## 3. Stock Volatility Distribution
Analyze return volatility profiles using boxplots.

In [ ]:
# Filter returns inside standard bounds for clean plotting
filtered_returns = combined_df[(combined_df['Daily_Return'] > -0.1) & (combined_df['Daily_Return'] < 0.1)]

sns.boxplot(x='Ticker', y='Daily_Return', data=filtered_returns)
plt.title("Equities Return Volatility Distributions (Outliers Clipped at ±10%)")
plt.ylabel("Daily Return")
plt.xlabel("Ticker")
plt.show()

## 4. Sector Correlation Matrix
Check how NASDAQ tech giants (AAPL, MSFT) correlate with auto disruptive innovators (TSLA) and NSE market indices.

In [ ]:
corr_matrix = pivot_close.corr()

sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Stock Price Correlation Heatmap")
plt.show()

## 5. Summary Insights
- **High Correlation**: Apple and Microsoft display strong technology-sector co-movements.
- **Beta Risk Profiles**: Tesla exhibits widest return distributions, signaling high historical beta volatility.
- **Indian Equities Trend**: Reliance and TCS show relative decoupling from US indexes during selective quarters, demonstrating diversification benefits.